# Бустинг

Будем предсказывать зарплату data scientist-ов в зависимости  от ряда факторов с помощью градиентного бустинга.

В датасете есть следующие признаки:



* work_year: The number of years of work experience in the field of data science.

* experience_level: The level of experience, such as Junior, Senior, or Lead.

* employment_type: The type of employment, such as Full-time or Contract.

* job_title: The specific job title or role, such as Data Analyst or Data Scientist.

* salary: The salary amount for the given job.

* salary_currency: The currency in which the salary is denoted.

* salary_in_usd: The equivalent salary amount converted to US dollars (USD) for comparison purposes.

* employee_residence: The country or region where the employee resides.

* remote_ratio: The percentage of remote work offered in the job.

* company_location: The location of the company or organization.

* company_size: The company's size is categorized as Small, Medium, or Large.

In [1]:
import pandas as pd

df = pd.read_csv("ds_salaries.csv")
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2023,SE,FT,Principal Data Scientist,80000,EUR,85847,ES,100,ES,L
1,2023,MI,CT,ML Engineer,30000,USD,30000,US,100,US,S
2,2023,MI,CT,ML Engineer,25500,USD,25500,US,100,US,S
3,2023,SE,FT,Data Scientist,175000,USD,175000,CA,100,CA,M
4,2023,SE,FT,Data Scientist,120000,USD,120000,CA,100,CA,M


### Подготовка данных

In [2]:
from sklearn.model_selection import train_test_split

data = df.drop(columns='salary')
X = data.drop(columns='salary_in_usd')
y = data['salary_in_usd']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

### Обучение линейной модели

In [3]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error
from sklearn.preprocessing import OneHotEncoder

categorical_features = list(X.select_dtypes(include=object).columns)

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_cat_encoded_train = pd.DataFrame(encoder.fit_transform(X_train[categorical_features]), columns=encoder.get_feature_names_out(categorical_features), index=X_train.index)
X_cat_encoded_val = pd.DataFrame(encoder.transform(X_val[categorical_features]), columns=encoder.get_feature_names_out(categorical_features), index=X_val.index)
X_cat_encoded_test = pd.DataFrame(encoder.transform(X_test[categorical_features]), columns=encoder.get_feature_names_out(categorical_features), index=X_test.index)

X_train_enc = pd.concat([X_train.drop(columns=categorical_features), X_cat_encoded_train], axis=1)
X_val_enc = pd.concat([X_val.drop(columns=categorical_features), X_cat_encoded_val], axis=1)
X_test_enc = pd.concat([X_test.drop(columns=categorical_features), X_cat_encoded_test], axis=1)

linreg = LinearRegression().fit(X_train_enc, y_train)
y_pred = linreg.predict(X_test_enc)

print('MAPE: ', mean_absolute_percentage_error(y_test, y_pred))
print('RMSE: ', np.sqrt(mean_squared_error(y_test, y_pred)))

MAPE:  0.3732694301523715
RMSE:  51564.895690004065


### XGboost

In [4]:
import itertools
from xgboost.sklearn import XGBRegressor

params = {
    'max_depth' : range(1, 10, 1),
    'learning_rate' : [0.05, 0.1, 0.3],
    'n_estimators' : range(50, 310, 50),
    'gamma' : np.arange(0.1, 0.6, 0.1)
}

params_comb = list(itertools.product(*params.values()))
best_rmse = 10**15

for par in params_comb:
    param = dict(zip(params.keys(), par))
    model_xgb = XGBRegressor(**param, random_state=42)
    model_xgb.fit(X_train_enc, y_train)
    y_val_pred = model_xgb.predict(X_val_enc)
    mape = mean_absolute_percentage_error(y_val, y_val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    if rmse < best_rmse:
        best_rmse = rmse
        best_params = param
best_params

{'max_depth': 2,
 'learning_rate': 0.3,
 'n_estimators': 100,
 'gamma': np.float64(0.1)}

In [5]:
import time

time_start_train = time.time()
model_xgb_best = XGBRegressor(**best_params, random_state=42)
model_xgb_best.fit(X_train_enc, y_train)
train_time = time.time() - time_start_train

time_start_pred = time.time()
y_pred = model_xgb_best.predict(X_test_enc)
pred_time = time.time() - time_start_pred

print('MAPE: ', mean_absolute_percentage_error(y_test, y_pred))
print('RMSE: ', np.sqrt(mean_squared_error(y_test, y_pred)))

print('Время обучения:', train_time)
print('Время предсказания:', pred_time)

MAPE:  0.3451322615146637
RMSE:  50404.0633282675
Время обучения: 0.220245361328125
Время предсказания: 0.01697993278503418


### CatBoost

In [6]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


In [7]:
from catboost import CatBoostRegressor

params = {
    'depth' : range(1, 10, 1),
    'learning_rate' : [0.05, 0.1, 0.3],
    'iterations' :  range(50, 310, 50)
}

params_comb = list(itertools.product(*params.values()))
best_rmse = 10**15

for par in params_comb:
    param = dict(zip(params.keys(), par))
    model_cat = CatBoostRegressor(**param, random_state=42, verbose=False)
    model_cat.fit(X_train_enc, y_train)
    y_val_pred = model_cat.predict(X_val_enc)
    mape = mean_absolute_percentage_error(y_val, y_val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    if rmse < best_rmse:
        best_rmse = rmse
        best_params = param
best_params

{'depth': 3, 'learning_rate': 0.3, 'iterations': 300}

In [8]:
time_start_train = time.time()
model_cat_best = CatBoostRegressor(**best_params, random_state=42, verbose=False)
model_cat_best.fit(X_train_enc, y_train)
train_time = time.time() - time_start_train

time_start_pred = time.time()
y_pred = model_cat_best.predict(X_test_enc)
pred_time = time.time() - time_start_pred

print('MAPE: ', mean_absolute_percentage_error(y_test, y_pred))
print('RMSE: ', np.sqrt(mean_squared_error(y_test, y_pred)))

print('Время обучения:', train_time)
print('Время предсказания:', pred_time)

MAPE:  0.345366424559636
RMSE:  50125.818359869154
Время обучения: 0.3100154399871826
Время предсказания: 0.006693601608276367


In [9]:
from catboost import Pool

X_train_pool = Pool(data=X_train, label=y_train, cat_features=categorical_features)
X_val_pool = Pool(data=X_val, cat_features=categorical_features)
X_test_pool = Pool(data=X_test, cat_features=categorical_features)

best_rmse = 10**9

for par in params_comb:
    param = dict(zip(params.keys(), par))
    model_cat = CatBoostRegressor(**param, random_state=42, verbose=False)
    model_cat.fit(X_train_pool)
    y_val_pred = model_cat.predict(X_val_pool)
    mape = mean_absolute_percentage_error(y_val, y_val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    if rmse < best_rmse:
        best_rmse = rmse
        best_params = param

print(best_params)
time_start_train = time.time()
model_cat_best = CatBoostRegressor(**best_params, random_state=42, verbose=False)
model_cat_best.fit(X_train_pool)
train_time = time.time() - time_start_train

time_start_pred = time.time()
y_pred = model_cat_best.predict(X_test_pool)
pred_time = time.time() - time_start_pred

print('MAPE: ', mean_absolute_percentage_error(y_test, y_pred))
print('RMSE: ', np.sqrt(mean_squared_error(y_test, y_pred)))

print('Время обучения:', train_time)
print('Время предсказания:', pred_time)

{'depth': 5, 'learning_rate': 0.1, 'iterations': 300}
MAPE:  0.36368088380404645
RMSE:  49572.92980590235
Время обучения: 1.5136325359344482
Время предсказания: 0.001529693603515625


### LightGBM

In [10]:
from lightgbm import LGBMRegressor


params = {
    'max_depth' : range(1, 10, 1),
    'learning_rate' : [0.05, 0.1, 0.3],
    'n_estimators' : range(50, 310, 50)
}

params_comb = list(itertools.product(*params.values()))
best_rmse = 10**9

for par in params_comb:
    param = dict(zip(params.keys(), par))
    model_lgbm = LGBMRegressor(**param, random_state=42, verbose=-1)
    model_lgbm.fit(X_train_enc, y_train)
    y_val_pred = model_lgbm.predict(X_val_enc)
    mape = mean_absolute_percentage_error(y_val, y_val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    if rmse < best_rmse:
        best_rmse = rmse
        best_params = param
best_params

{'max_depth': 1, 'learning_rate': 0.3, 'n_estimators': 150}

In [11]:
time_start_train = time.time()
model_cat_best = LGBMRegressor(**best_params, random_state=42, verbose=-1)
model_cat_best.fit(X_train_enc, y_train)
train_time = time.time() - time_start_train

time_start_pred = time.time()
y_pred = model_cat_best.predict(X_test_enc)
pred_time = time.time() - time_start_pred

print('MAPE: ', mean_absolute_percentage_error(y_test, y_pred))
print('RMSE: ', np.sqrt(mean_squared_error(y_test, y_pred)))

print('Время обучения:', train_time)
print('Время предсказания:', pred_time)

MAPE:  0.3585805902656797
RMSE:  50562.1830241073
Время обучения: 0.0295407772064209
Время предсказания: 0.003222227096557617


Самые лучшие показатели по ошибке RMSE показала модель CatBoostRegressor при обучении с использованием Pool, лучшие показатели по MAPE у XGBRegressor. Дольше всего обучался CatBoostRegressor при использованием Pool - зато он же предсказывал быстрее других.

Лучший гиперпараметр max depth для LGBMRegressor - 1, для XGBRegressor - 2, depth для CatBoostRegressor - 3, при использовании Pool - 5,  это подтверждает, что для бустинга лучше использовать не глубокие деревья. Глубокие деревья в градиентном бустинге могут приводить к переобучению. Для гиперпараметра learning rate наилучшим для всех моделей оказался 0.3 (кроме CatBoostRegressor при использовании Pool - 0.1). Лучшим числом деревьев n_estimators для XGBRegressor оказалось 100, LGBMRegressor - 150, iterations для CatBoostRegressor - 300, при использовании Pool - 300. Таким образом, CatBoostRegressor нужно больше деревьев.

Модель линейной регрессии показала худшие показатели по MAPE и RMSE по сравнению со всеми моделями градбустинга